In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import io
import os

os.chdir("C:/Users/SEBASTIAN/Documents/Cursos/curso_ciencia_de_datos/30_days_of_python/Inferencia_estadistica/")

econdata= pd.read_csv("Econdata.csv")
econdata.head(2)

,id,geo_point_2d,geo_shape,clave_cat,delegacion,perimetro,tipo,nom_id
0,0,"19.424781053,-99.1327537959","{""type"": ""Polygon"", ""coordinates"": [[[-99.1332...",307_130_11,Cuauhtémoc,B,Mercado,Pino Suárez
1,1,"19.4346139576,-99.1413808393","{""type"": ""MultiPoint"", ""coordinates"": [[-99.14...",002_008_01,Cuautémoc,A,Museo,Museo Nacional de Arquitectura Palacio de Bell...


In [ ]:
#FUNCIONES DE MUESTREO

#Muestreo aleatorio simple
aleat_8= econdata.sample(n=8) #Tomar 8 datos de manera aleatoria
prop_25= econdata.sample(frac= 0.25) #Tomar el 25% de los datos de manera aleatoria

#Muestreo sistematico

def systematic_sampling(data, step): #Crear una funcion para hacer un muestreo aleatorio sistematico                                                                                      
    indexes= np.arange(0, len(data), step= step)
    systematic_sample= data.iloc[indexes]
    return systematic_sample

systematic_sample= systematic_sampling(econdata, 3) #Muestreo aleatorio sistematico de 3 en 3 en todos los datos

#Muestreo aleatorio estratificado
econdata["estratificado"] = econdata["delegacion"] + "," + econdata["tipo"] #Crear una nueva variable en el df
(econdata["estratificado"].value_counts()/len(econdata)).sort_values(ascending= False)

#Imagina que nuestros datos son diferentes y sabemos que la proporción es la siguiente:

#1) Hoteles en Cuauhtémoc es de 0.5
#2) Museos en Cuauhtémoc es de 0.2
#3) Hoteles en Venuestiano Carranza es de 0.1
#4) Mercados en Cuauhtémoc es de 0.1
#5) Mercados en Venuestiano Carranza es de 0.1

#Queremos una nueva tabla e indicar como queremos que sea el muestreo estratificado

def data_estratificada(econdata, nombres_columnas_estrat, valores_estrat, prop_estrat, random_state=None):
    # Crea un DataFrame vacío con las mismas columnas que econdata
    df_estrat = pd.DataFrame(columns = econdata.columns) 

    pos = -1
    for i in range(len(valores_estrat)): # Itera sobre los valores estratificados
        pos += 1
        if pos == len(valores_estrat) - 1: 
            # Si es la última iteración, calcula el número de valores de salida para que tenga el mismo número de filas que de entrada
            ratio_len = len(econdata) - len(df_estrat) 
        else:
            # Calcula el número de filas según la proporción deseada
            ratio_len = int(len(econdata) * prop_estrat[i])

        # Filtra los datos de origen según los valores seleccionados en la estratificación
        df_filtrado = econdata[econdata[nombres_columnas_estrat] ==valores_estrat[i]] 
        # Realiza un muestreo aleatorio de los datos filtrados usando la ratio calculada
        df_temp = df_filtrado.sample(replace=True, n=ratio_len, random_state=random_state)
        
        # Concatena el DataFrame estratificado con los datos muestreados
        df_estrat = pd.concat([df_estrat, df_temp])
        
    return df_estrat # Devuelve los datos re-muestreados y estratificados

# Valores y proporciones para la estratificación
valores_estrat = ['Cuautémoc,Hotel', 'Cuautémoc,Museo', 'Venustiano Carranza,Hotel', 'Cuauhtémoc,Mercado','Venustiano Carranza,Mercado']
prop_estrat = [0.5, 0.2, 0.1, 0.1,0.1]
# Llama a la función de estratificación y guarda el resultado en df_estrat
df_estrat = data_estratificada(econdata, 'estratificado', valores_estrat, prop_estrat, random_state=42)
df_estrat
